# Notebook 03: Cross-Validation Against Existing Footprint Products

**Inputs:** 3-class prediction maps (Notebook 02), Google Open Buildings, Microsoft GlobalML, OpenBuildingMap  
**Outputs:** Per-500m-cell agreement metrics, class-specific disagreement surfaces, spatial bias gradient

---

In [ ]:
import json, os
import numpy as np
import geopandas as gpd
import rasterio
from rasterio.transform import from_bounds
from rasterio.features import rasterize
import pandas as pd
import matplotlib.pyplot as plt

with open('../data/config.json') as f:
    CONFIG = json.load(f)

print('Libraries loaded.')

## 3.1 Load Prediction Maps and Footprint Products

In [ ]:
# -------------------------------------------------------
# EDIT THESE: update paths once predictions are exported
PREDICTION_PATHS = {
    c: f'../outputs/predictions_{c.lower()}.tif'
    for c in CONFIG['countries']
}

FOOTPRINT_PATHS = {
    'google_ob':       '../data/footprints/google_open_buildings_5country.gpkg',
    'microsoft_globalml': '../data/footprints/microsoft_globalml_5country.gpkg',
    'open_building_map':  '../data/footprints/open_building_map_5country.gpkg',
}
# -------------------------------------------------------

print('Data paths configured.')
print('Prediction paths:', PREDICTION_PATHS)
print('Footprint paths:', FOOTPRINT_PATHS)

## 3.2 Aggregate Predictions to 500m Grid

In [ ]:
def aggregate_to_500m(pred_path, resolution_m=500):
    """
    Aggregates 10m prediction raster to 500m grid cells.
    Returns dict of per-cell statistics:
      - n_background, n_isolated, n_clustered: pixel counts per class
      - field_density: fraction of cell classified as settled
      - iso_to_cluster_ratio: isolated / clustered pixel ratio
      - entropy_isolated, entropy_clustered: mean model entropy per class
    """
    with rasterio.open(pred_path) as src:
        pred = src.read(1)  # class map
        # band 2 and 3 would carry entropy if exported
        transform = src.transform
        crs = src.crs

    scale_factor = resolution_m // 10  # 50x50 native pixels per 500m cell
    H, W = pred.shape
    H_out = H // scale_factor
    W_out = W // scale_factor

    # Truncate to divisible size
    pred = pred[:H_out * scale_factor, :W_out * scale_factor]
    pred_blocks = pred.reshape(H_out, scale_factor, W_out, scale_factor)

    n_background = (pred_blocks == 0).sum(axis=(1, 3))
    n_isolated   = (pred_blocks == 1).sum(axis=(1, 3))
    n_clustered  = (pred_blocks == 2).sum(axis=(1, 3))
    total        = scale_factor ** 2

    field_density = (n_isolated + n_clustered) / total

    with np.errstate(divide='ignore', invalid='ignore'):
        iso_to_cluster = np.where(
            n_clustered > 0, n_isolated / n_clustered, np.nan
        )

    return {
        'n_isolated':        n_isolated,
        'n_clustered':       n_clustered,
        'n_background':      n_background,
        'field_density':     field_density,
        'iso_to_cluster':    iso_to_cluster,
        'H_out': H_out, 'W_out': W_out,
        'transform': transform, 'crs': crs
    }


cell_stats = {}
for country in CONFIG['countries']:
    path = PREDICTION_PATHS[country]
    if os.path.exists(path):
        cell_stats[country] = aggregate_to_500m(path)
        print(f'{country}: aggregated to {cell_stats[country]["H_out"]}x{cell_stats[country]["W_out"]} 500m cells')
    else:
        print(f'WARNING: {path} not found — run Notebook 02 first')

## 3.3 Compute Agreement Rate Against Each Footprint Product

In [ ]:
def rasterize_footprints(footprint_gdf, ref_raster_path, resolution_m=500):
    """
    Rasterizes building footprint polygons onto the 500m grid.
    Returns binary array: 1 = footprint present, 0 = absent.
    """
    with rasterio.open(ref_raster_path) as src:
        transform = src.transform
        shape = (src.height // 50, src.width // 50)  # 500m cells
        crs = src.crs

    # Reproject footprints to match raster CRS
    footprint_gdf = footprint_gdf.to_crs(crs)

    # Rasterize
    burned = rasterize(
        [(geom, 1) for geom in footprint_gdf.geometry],
        out_shape=shape,
        transform=transform * rasterio.Affine.scale(50),  # scale to 500m
        fill=0,
        dtype='uint8'
    )
    return burned


def compute_agreement_rate(pred_settled_mask, footprint_raster):
    """
    Computes fraction of predicted settled cells that overlap with footprint product.
    pred_settled_mask: binary 2D array (1 = model detects settlement)
    footprint_raster:  binary 2D array (1 = footprint product has building)
    Returns per-cell agreement rate (0-1).
    """
    agreement = (pred_settled_mask & footprint_raster).astype(float)
    denominator = pred_settled_mask.astype(float)
    with np.errstate(divide='ignore', invalid='ignore'):
        rate = np.where(denominator > 0, agreement / denominator, np.nan)
    return rate


print('Agreement rate functions defined.')
print('Load footprint GeoDataFrames and call compute_agreement_rate() per product.')

## 3.4 Product Consensus Count

In [ ]:
def compute_consensus_count(footprint_rasters):
    """
    Computes per-cell count of how many footprint products agree a building is present.
    footprint_rasters: list of binary 2D arrays, one per product
    Returns integer array 0-3.
    """
    stacked = np.stack(footprint_rasters, axis=0)  # (n_products, H, W)
    consensus = stacked.sum(axis=0)                # (H, W), values 0-3
    return consensus


print('Consensus count function defined.')
print('Call with list of [google_ob_raster, microsoft_raster, obmap_raster] per country.')

## 3.5 Temporal Stability Score

In [ ]:
def compute_temporal_stability(pred_dry_path, pred_wet_path):
    """
    Computes fraction of settled pixels detected consistently in both dry and wet composites.
    High stability = stronger evidence of true settlement.
    Returns 500m-resolution stability score per cell.
    """
    dry_stats  = aggregate_to_500m(pred_dry_path)
    wet_stats  = aggregate_to_500m(pred_wet_path)

    dry_settled = (dry_stats['n_isolated'] + dry_stats['n_clustered']) > 0
    wet_settled = (wet_stats['n_isolated'] + wet_stats['n_clustered']) > 0

    # Stability = fraction of union that is intersection
    intersection = (dry_settled & wet_settled).astype(float)
    union = (dry_settled | wet_settled).astype(float)
    with np.errstate(divide='ignore', invalid='ignore'):
        stability = np.where(union > 0, intersection / union, 0.0)

    return stability


print('Temporal stability function defined.')
print('Requires dry and wet season prediction maps exported separately from Notebook 02.')

## 3.6 Spatial Bias Gradient

Quantifies how agreement rate varies with distance from urban centres and roads.

In [ ]:
import ee, json
ee.Initialize(project=CONFIG['gee_project'])

def export_distance_layers(study_area_geometry, export_folder):
    """
    Exports two distance layers from GEE:
      1. Distance to nearest urban centre (from GHS-SMOD urban class)
      2. Distance to nearest road (from OpenStreetMap via GEE community dataset)
    Both at 500m resolution over the study area.
    """
    # Urban centre distance
    smod = ee.Image('JRC/GHSL/P2023A/GHS_SMOD/2020').select('smod_code')
    urban_mask = smod.gte(23)  # urban classes
    dist_urban = urban_mask.Not().cumulativeCost(
        source=urban_mask,
        maxDistance=200000  # 200km max
    ).rename('dist_urban_m')

    task_urban = ee.batch.Export.image.toDrive(
        image=dist_urban.clip(study_area_geometry),
        description='dist_urban_500m',
        folder=export_folder,
        fileNamePrefix='dist_urban_500m',
        scale=500,
        region=study_area_geometry,
        maxPixels=1e10,
        fileFormat='GeoTIFF',
        formatOptions={'cloudOptimized': True}
    )
    task_urban.start()
    print(f'Urban distance export submitted: task ID {task_urban.id}')
    return task_urban


print('Distance layer export function defined.')
print('Call export_distance_layers(study_area, CONFIG["gee_export_folder"]) to submit task.')

## 3.7 Assemble Feature Matrix

Combines all Stage 3 metrics into the feature vector for the Stage 4 Random Forest.

In [ ]:
def assemble_feature_matrix(country, cell_stats, agreement_rates,
                             consensus, temporal_stability,
                             dist_urban, dist_road,
                             entropy_isolated=None, entropy_clustered=None):
    """
    Assembles the 12-feature input vector per 500m cell for the confidence Random Forest.

    Features:
      0: entropy_isolated
      1: entropy_clustered
      2: n_isolated (pixel count)
      3: n_clustered
      4: field_density
      5: iso_to_cluster_ratio
      6: agreement_rate_google_ob
      7: agreement_rate_microsoft
      8: agreement_rate_obmap
      9: product_consensus_count
      10: temporal_stability
      11: dist_urban_m
    """
    stats = cell_stats[country]
    H, W = stats['H_out'], stats['W_out']

    feature_matrix = np.stack([
        entropy_isolated if entropy_isolated is not None else np.zeros((H, W)),
        entropy_clustered if entropy_clustered is not None else np.zeros((H, W)),
        stats['n_isolated'].astype(float),
        stats['n_clustered'].astype(float),
        stats['field_density'],
        np.nan_to_num(stats['iso_to_cluster'], nan=0.0),
        agreement_rates.get('google_ob', np.zeros((H, W))),
        agreement_rates.get('microsoft', np.zeros((H, W))),
        agreement_rates.get('obmap',     np.zeros((H, W))),
        consensus.astype(float),
        temporal_stability,
        dist_urban,
    ], axis=-1)  # (H, W, 12)

    # Flatten to (n_cells, 12)
    n_cells = H * W
    X = feature_matrix.reshape(n_cells, 12)
    print(f'{country}: feature matrix shape {X.shape}')
    return X


print('Feature matrix assembly function defined.')
print('Call once all Stage 3 metrics are computed per country.')

---
## Stage 3 Complete

Proceed to **Notebook 04: Confidence Surface and Population Bias Quantification**.

**Files produced:**
- `../outputs/feature_matrix_{country}.npy` — (n_cells, 12) feature arrays
- `../outputs/consensus_{country}.tif` — product consensus count raster
- `../outputs/agreement_{country}.tif` — per-product agreement rate rasters